# KarstGeophysics Project

## Geode SEG-2 shot gathers, SEG-Y export, wiggle plots, and Charlie-style frequency contours

This notebook reads Geode SEG-2 (`*.dat`) shot gathers, assigns provisional geometry, writes SEG-Y files, and creates:

1. wiggle plots for each shot gather;
2. Charlie-style frequency-vs-offset contour plots; and
3. receiver-coordinate frequency-contour plots for cave-map interpretation.

The frequency plots use a simple FFT-per-trace workflow that follows Charlie Breithaupt's frequency-vs-offset plotting more closely than the earlier generic spectral plot: each trace is demeaned, normalized by its peak-to-peak range, transformed to the frequency domain, then plotted against offset or receiver coordinate.


In [1]:
from pathlib import Path
from datetime import date
import sys
from typing import Literal, Sequence

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
import obspy
from obspy import Stream


def find_repo_root(start: Path, required_subdir: str = "lib") -> Path:
    """Find the repository root by walking upward until ``required_subdir`` is found.

    This lets the notebook live anywhere under the repository while importing
    packages from the top-level ``lib/`` directory.
    """
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / required_subdir).is_dir():
            return candidate
    raise FileNotFoundError(
        f"Could not find a repository root containing {required_subdir!r} above {start}. "
        "Set repo_root manually."
    )


repo_root = find_repo_root(Path.cwd())
lib_dir = repo_root / "lib"
if str(lib_dir) not in sys.path:
    sys.path.insert(0, str(lib_dir))

print(f"repo_root = {repo_root}")
print(f"lib_dir   = {lib_dir}")

from segy_tools.gather import (
    plot_wiggle_gather_from_stream,
    stream_to_gather_arrays,
)
from segy_tools.io import write_segy



repo_root = /Users/glennthompson/Developer/karstgeo
lib_dir   = /Users/glennthompson/Developer/karstgeo/lib


## User configuration

Update these values for the local machine and the specific subset of field folders to process.

The notebook handles two folder-name conventions used so far:

- `LBS_*`
- `LBSSP_*`

The processing function below can be run for either or both patterns.


In [2]:
DOWNLOAD_DIR = Path("~/Downloads").expanduser()

# Excel workbook containing Jochen's digitized field-note metadata tables.
# The notebook will look this up relative to the repository first, then fall back
# to the notebook working directory / explicit path if needed.
box_folder_root = Path("/Users/glennthompson/Library/CloudStorage/Box-Box/thompsong/2026KarstGeophysicsDEP")
seismic_analysis_dir = box_folder_root / "04_FinalReport" / "seismic_analysis"
METADATA_XLSX = seismic_analysis_dir /  "jochen_field_notes_metadata_tables.xlsx"

# Sheets to search for Geode SEG-2 file numbers. Each sheet should contain a
# file_no column. File numbers are matched against datfile.stem, e.g. 3046.dat.
METADATA_SHEETS = [
    "T1_1m_Refraction",
    "T1_2m_Refraction",
    "T1_Streamer_MASW",
    "T1A_Streamer_MASW",
    "T3_1m_Refraction",
    "T4_1m_Refraction",
]

# Folder name patterns to process. Use ["LBSSP_"] for the current Lafayette Blue Springs folders,
# or include "LBS_" if older folders should also be processed.
FOLDER_PREFIXES = ["LBSSP_"]

# Plotting / processing controls.
feet_to_meters = 0.3048
wiggle_scale = 1.0
normalize_wiggles = True

# Optional bandpass before wiggle plotting. Keep False for raw plots.
APPLY_BANDPASS_FOR_WIGGLES = False
BANDPASS_FREQ_HZ = (200.0, 400.0)

# Frequency-contour controls.
MAKE_FREQUENCY_CONTOURS = True
FREQ_CONTOUR_MAX_HZ = 100.0       # Charlie-style plots commonly used 0--100 Hz.
FREQ_CONTOUR_LEVELS = 14
FREQ_CONTOUR_PLOT_TYPE_CHARLIE = "contour"      # contour, contourf, or pcolormesh
FREQ_CONTOUR_PLOT_TYPE_MAP = "contourf"         # contourf/pcolormesh is better for map-style plots

# Set to a tuple if you want to mark suspected cave center(s), e.g. (122.0, 130.0).
CAVE_MARKERS_M = (122.0, 130.0)

# File writing controls.
WRITE_SEGY = True
OVERWRITE_PLOTS = True
OVERWRITE_SEGY = False


### Metadata-driven geometry

This notebook now uses Jochen's digitized field-note workbook as the authoritative geometry lookup.  
Each SEG-2 file is matched by `file_no` (`datfile.stem`) against the configured Excel sheets. Geometry corrections should therefore be made in the spreadsheet rather than hard-coded in this notebook.


## Helper functions

These are project-specific wrappers around `segy_tools`. They mostly encode current field geometry assumptions and folder/file naming conventions. Once the final acquisition tables are settled, these geometry rules should be replaced by metadata-table lookups rather than filename or duration heuristics.


In [3]:
def parse_date_from_geode_folder(folder: Path) -> date | None:
    """Parse a date from a Geode download folder name.

    Expected examples include names like ``LBSSP_MMDDYY_*`` or ``LBS_MMDDYY_*``.
    Returns ``None`` if the folder name does not contain a parseable date token.
    """
    parts = folder.name.split("_")
    if len(parts) < 2:
        return None
    mdy = parts[1]
    if len(mdy) != 6 or not mdy.isdigit():
        return None
    return date(2000 + int(mdy[4:6]), int(mdy[0:2]), int(mdy[2:4]))


def find_geode_folders(download_dir: Path, prefixes=("LBSSP_",)) -> list[Path]:
    """Return Geode data folders whose names start with any requested prefix."""
    folders = []
    for item in sorted(download_dir.iterdir()):
        if item.is_dir() and any(item.name.startswith(prefix) for prefix in prefixes):
            folders.append(item)
    return folders


def resolve_metadata_workbook(path: Path) -> Path:
    """Resolve the metadata workbook path.

    The recommended location is ``repo_root/metadata/jochen_field_notes_metadata_tables.xlsx``.
    If that path does not exist, this function also checks the current working
    directory and the directory containing the notebook output during development.
    """
    candidates = [
        Path(path).expanduser(),
        Path.cwd() / Path(path).name,
        repo_root / Path(path).name,
        repo_root / "metadata" / Path(path).name,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        "Could not find metadata workbook. Checked:\n"
        + "\n".join(f"  - {c}" for c in candidates)
    )


def _clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy with stripped, lower-case, underscore-separated column names."""
    out = df.copy()
    out.columns = [
        str(c).strip().lower().replace(" ", "_").replace("-", "_")
        for c in out.columns
    ]
    return out


def _none_if_nan(value):
    """Convert pandas/NumPy missing values to None."""
    return None if pd.isna(value) else value


def load_geode_metadata_lookup(
    metadata_xlsx: Path,
    sheet_names: list[str] | tuple[str, ...] = METADATA_SHEETS,
) -> pd.DataFrame:
    """Load Jochen field-note metadata tables into one file-number lookup table.

    Parameters
    ----------
    metadata_xlsx
        Excel workbook containing one acquisition table per sheet.

    sheet_names
        Sheet names to scan. Each useful sheet must contain a ``file_no`` column.

    Returns
    -------
    pandas.DataFrame
        Combined metadata table with a ``sheet_name`` column. File numbers are
        stored as integers in the ``file_no`` column.

    Notes
    -----
    The intent is that geometry changes are made in the spreadsheet, not in the
    notebook. This function only standardizes common column names and combines
    sheets into a single lookup table.
    """
    metadata_xlsx = resolve_metadata_workbook(metadata_xlsx)
    rows = []

    xls = pd.ExcelFile(metadata_xlsx)
    available = set(xls.sheet_names)

    for sheet_name in sheet_names:
        if sheet_name not in available:
            print(f"WARNING: metadata sheet not found, skipping: {sheet_name}")
            continue

        df = pd.read_excel(metadata_xlsx, sheet_name=sheet_name)
        df = _clean_column_names(df)

        if "file_no" not in df.columns:
            print(f"WARNING: sheet has no file_no column, skipping: {sheet_name}")
            continue

        df = df[pd.notna(df["file_no"])].copy()
        if df.empty:
            continue

        df["file_no"] = df["file_no"].astype(int)
        df["sheet_name"] = sheet_name

        # Standardize common source-position aliases.
        if "source_position_m" not in df.columns:
            for alias in ("shot_location_m", "shot_position_m", "source_x_m"):
                if alias in df.columns:
                    df["source_position_m"] = df[alias]
                    break

        rows.append(df)

    if not rows:
        raise ValueError(f"No usable metadata rows found in {metadata_xlsx}")

    meta = pd.concat(rows, ignore_index=True, sort=False)

    # Ensure standardized columns exist even where not applicable.
    for col in [
        "source_position_m",
        "receiver_first_m",
        "receiver_last_m",
        "receiver_spacing_m",
        "transect",
        "survey",
        "operator",
        "plate_type",
        "n_blows",
        "n_shots",
        "confidence",
        "review_status",
        "comment",
        "comments",
    ]:
        if col not in meta.columns:
            meta[col] = np.nan

    # Warn about duplicate file numbers, but allow disambiguation by folder/sheet later if desired.
    duplicates = meta.loc[meta["file_no"].duplicated(keep=False), ["file_no", "sheet_name"]]
    if len(duplicates):
        print("WARNING: duplicate file_no values found in metadata lookup:")
        display(duplicates.sort_values("file_no").head(50))

    print(f"Loaded {len(meta)} metadata rows from {metadata_xlsx}")
    return meta


def lookup_geode_metadata(file_no: int | str, metadata_lookup: pd.DataFrame) -> dict:
    """Return one metadata row matching a Geode file number.

    Parameters
    ----------
    file_no
        SEG-2 file number, usually ``int(datfile.stem)``.

    metadata_lookup
        Combined table from :func:`load_geode_metadata_lookup`.

    Returns
    -------
    dict
        Metadata row as a plain dictionary, with missing values converted to None.

    Raises
    ------
    KeyError
        If the file number is absent or ambiguous.
    """
    file_no = int(file_no)
    matches = metadata_lookup.loc[metadata_lookup["file_no"] == file_no].copy()

    if len(matches) == 0:
        raise KeyError(f"No metadata found for file_no={file_no}")

    if len(matches) > 1:
        summary = matches[["file_no", "sheet_name"]].to_dict("records")
        raise KeyError(f"Multiple metadata rows found for file_no={file_no}: {summary}")

    row = matches.iloc[0].to_dict()
    return {key: _none_if_nan(value) for key, value in row.items()}


def metadata_to_geometry(meta: dict, st: Stream) -> dict:
    """Convert one spreadsheet metadata row into the geometry dictionary used downstream.

    Parameters
    ----------
    meta
        Metadata dictionary returned by :func:`lookup_geode_metadata`.

    st
        Raw ObsPy Stream read from the SEG-2 file.

    Returns
    -------
    dict
        Geometry dictionary compatible with plotting/export functions.

    Notes
    -----
    Refraction sheets should ideally provide ``receiver_first_m``,
    ``receiver_last_m`` and ``receiver_spacing_m`` directly. Streamer/MASW sheets
    may lack receiver coordinates; in that case a documented provisional fallback
    assumes 24 channels, 5-ft spacing, and the first active receiver approximately
    30 ft behind the PEG source.
    """
    n_traces = len(st)
    duration_s = max(tr.stats.npts * tr.stats.delta for tr in st)

    sheet_name = meta.get("sheet_name")
    survey = meta.get("survey")
    transect = meta.get("transect")

    source_x_m = meta.get("source_position_m")
    receiver_first_m = meta.get("receiver_first_m")
    receiver_last_m = meta.get("receiver_last_m")
    receiver_spacing_m = meta.get("receiver_spacing_m")

    if source_x_m is None:
        raise ValueError(f"No source_position_m for file_no={meta.get('file_no')} in {sheet_name}")

    # Streamer/MASW fallback when receiver geometry is absent.
    if receiver_first_m is None and sheet_name in {"T1_Streamer_MASW", "T1A_Streamer_MASW"}:
        receiver_spacing_m = 5.0 * feet_to_meters
        # Earlier field notes: first active receiver roughly 30 ft behind PEG.
        receiver_first_m = float(source_x_m) - 30.0 * feet_to_meters
        receiver_last_m = receiver_first_m - (n_traces - 1) * receiver_spacing_m

    # If first receiver is missing but last and spacing exist, infer first.
    if receiver_first_m is None and receiver_last_m is not None and receiver_spacing_m is not None:
        receiver_first_m = float(receiver_last_m) - (n_traces - 1) * float(receiver_spacing_m)

    # If receiver_last is missing but first and spacing exist, infer last.
    if receiver_last_m is None and receiver_first_m is not None and receiver_spacing_m is not None:
        receiver_last_m = float(receiver_first_m) + (n_traces - 1) * float(receiver_spacing_m)

    if receiver_spacing_m is None:
        raise ValueError(f"No receiver_spacing_m for file_no={meta.get('file_no')} in {sheet_name}")

    if receiver_first_m is None:
        raise ValueError(f"No receiver_first_m could be inferred for file_no={meta.get('file_no')} in {sheet_name}")

    network = "GV" if n_traces == 24 else "LB" if n_traces == 72 else "XX"
    geometry_name = survey or sheet_name or "metadata geometry"

    line_min = min(float(receiver_first_m), float(receiver_last_m), float(source_x_m))
    line_max = max(float(receiver_first_m), float(receiver_last_m), float(source_x_m))

    # For receiver-x cave-map plots, show at least the active spread and source.
    line_xlim_m = (line_min, line_max)

    return {
        "file_no": int(meta["file_no"]),
        "sheet_name": sheet_name,
        "transect": transect,
        "survey": survey,
        "geometry_name": geometry_name,
        "network": network,
        "n_traces": n_traces,
        "duration_s": duration_s,
        "receiver_spacing_m": float(receiver_spacing_m),
        "first_receiver_x_m": float(receiver_first_m),
        "last_receiver_x_m": float(receiver_last_m),
        "source_x_m": float(source_x_m),
        "line_xlim_m": line_xlim_m,
        "operator": meta.get("operator"),
        "plate_type": meta.get("plate_type"),
        "n_blows": meta.get("n_blows") or meta.get("n_shots"),
        "confidence": meta.get("confidence"),
        "review_status": meta.get("review_status"),
        "comment": meta.get("comment") or meta.get("comments"),
        "metadata": meta,
    }


def annotate_geode_stream(st: Stream, geom: dict) -> Stream:
    """Assign SEED-style metadata to a Geode SEG-2 stream.

    Geometry values are still supplied as fallback coordinates to `segy_tools`
    plotting and export functions. This function sets useful trace IDs for
    browsing and export.
    """
    st = st.copy()
    for i, tr in enumerate(st):
        tr.stats.station = f"CHA{i + 1:02d}"
        tr.stats.network = geom["network"]
        tr.stats.channel = "DHZ" if tr.stats.sampling_rate < 1000 else "GHZ"
        tr.stats.location = str(geom.get("file_no", 0))[-2:].zfill(2)
    return st


def maybe_bandpass(st: Stream, freqmin: float, freqmax: float) -> Stream:
    """Return a detrended, zero-phase bandpass-filtered copy of a stream."""
    stf = st.copy()
    stf.detrend(type="demean")
    stf.filter("bandpass", freqmin=freqmin, freqmax=freqmax, corners=4, zerophase=True)
    return stf


## Frequency-contour plotting

For each shot gather, the notebook now generates two frequency-domain quick-look plots:

1. **Charlie-style offset plot** — frequency vs signed source-receiver offset. This is closest to Charlie Breithaupt's frequency-vs-offset workflow.
2. **Cave-map receiver-x plot** — frequency vs receiver coordinate along the profile, with suspected cave-center markers shown where relevant.

The FFT workflow follows Charlie's simpler frequency-offset code closely: demean each trace, normalize each trace by its peak-to-peak range, compute a one-sided FFT amplitude spectrum per trace, then normalize the spectral image globally before plotting.


In [4]:
def plot_frequency_contour_from_stream_charlie_fft(
    st: Stream,
    *,
    fallback_receiver_spacing_m: float,
    fallback_first_receiver_x_m: float,
    fallback_source_x_m: float,
    receiver_x_m_override: Sequence[float] | None = None,
    source_x_m_override: float | None = None,
    max_freq_hz: float = 100.0,
    x_axis: Literal["offset", "receiver_x", "absolute_offset", "trace"] = "offset",
    line_xlim_m: tuple[float, float] | None = None,
    trace_scale_m: float | None = None,
    levels: int | Sequence[float] = 14,
    plot_type: Literal["contour", "contourf", "pcolormesh"] = "contour",
    title: str | None = None,
    outfile: Path | None = None,
    close_after_save: bool = False,
    cave_markers_m: tuple[float, ...] = (),
    cmap: str = "viridis",
) -> dict:
    """Create a Charlie-style FFT frequency-vs-position plot from an ObsPy Stream.

    This reproduces the spirit of Charlie Breithaupt's simpler frequency-vs-offset
    workflow:

    1. convert an ObsPy Stream to gather arrays;
    2. force the data to shape ``(n_traces, n_samples)``;
    3. demean each trace;
    4. normalize each trace by its peak-to-peak range, scaled by receiver spacing;
    5. compute an FFT amplitude spectrum for each trace;
    6. normalize the full gather spectrum globally;
    7. plot frequency versus offset or receiver coordinate.

    Parameters
    ----------
    st
        Shot gather as an ObsPy Stream.
    fallback_receiver_spacing_m, fallback_first_receiver_x_m, fallback_source_x_m
        Geometry values used if trace headers do not contain usable coordinates.
    receiver_x_m_override
        Optional explicit receiver coordinates for this gather. Use this when
        SEG-Y/SEG-2 headers are missing, wrong, or only contain relative positions.
    source_x_m_override
        Optional explicit source coordinate for this gather.
    max_freq_hz
        Maximum frequency to show on the vertical axis.
    x_axis
        Plot coordinate system: ``"offset"`` for signed source-receiver offset,
        ``"receiver_x"`` for line coordinate, ``"absolute_offset"`` for absolute
        source-receiver offset, or ``"trace"`` for trace number.
    line_xlim_m
        Optional x-axis limits. Useful for receiver-coordinate plots where the
        full spread or line extent should be visible.
    trace_scale_m
        Scale factor for Charlie-style trace normalization. Defaults to receiver
        spacing, matching the original MATLAB logic ``data = dx * data / range``.
    levels
        Number of contour levels or explicit contour levels.
    plot_type
        ``"contour"`` for line contours, ``"contourf"`` for filled contours,
        or ``"pcolormesh"`` for image-style plotting.
    title
        Optional plot title.
    outfile
        Optional output image path.
    close_after_save
        Close the figure after saving.
    cave_markers_m
        Cave/reference marker positions in receiver-x coordinates. These are
        converted to offset if ``x_axis`` is offset-based.
    cmap
        Matplotlib colormap name.

    Returns
    -------
    dict
        Dictionary containing the plotted arrays, spectra, geometry, figure,
        axis, and colorbar handles.
    """
    if len(st) == 0:
        raise ValueError("Input Stream is empty.")

    time, data, receiver_x_m, source_x_m, geom = stream_to_gather_arrays(
        st,
        sort_by="receiver_x",
        fallback_receiver_spacing_m=fallback_receiver_spacing_m,
        fallback_first_receiver_x_m=fallback_first_receiver_x_m,
        fallback_source_x_m=fallback_source_x_m,
    )

    data = np.asarray(data, dtype=float)
    time = np.asarray(time, dtype=float)
    receiver_x_m = np.asarray(receiver_x_m, dtype=float)

    if receiver_x_m_override is not None:
        receiver_x_m = np.asarray(receiver_x_m_override, dtype=float)

    if source_x_m_override is not None:
        source_x_m = float(source_x_m_override)

    if time.size < 2:
        raise ValueError("Time vector must contain at least two samples.")

    dt = float(np.median(np.diff(time)))
    if not np.isfinite(dt) or dt <= 0:
        raise ValueError("Invalid sample interval.")

    # Standardize data to (n_traces, n_samples).
    if data.shape[0] == receiver_x_m.size:
        working = data.copy()
    elif data.shape[1] == receiver_x_m.size:
        working = data.T.copy()
    else:
        raise ValueError(
            f"Cannot match data shape {data.shape} to "
            f"{receiver_x_m.size} receiver positions."
        )

    # Sort traces by receiver coordinate so contouring behaves correctly.
    idx = np.argsort(receiver_x_m)
    receiver_x_m = receiver_x_m[idx]
    working = working[idx, :]

    # Charlie-style demean and per-trace range normalization.
    working = working - np.nanmean(working, axis=1, keepdims=True)

    if trace_scale_m is None:
        trace_scale_m = fallback_receiver_spacing_m

    trange = np.nanmax(working, axis=1, keepdims=True) - np.nanmin(
        working, axis=1, keepdims=True
    )
    trange[~np.isfinite(trange)] = 1.0
    trange[trange == 0] = 1.0
    working = trace_scale_m * working / trange

    freqs = np.fft.rfftfreq(working.shape[1], d=dt)
    spec = np.abs(np.fft.rfft(working, axis=1))  # (n_traces, n_freq)

    keep = freqs <= max_freq_hz
    freqs = freqs[keep]
    spec = spec[:, keep]

    # Plot as (n_freq, n_traces), matching matplotlib contour expectations.
    spectrum = spec.T

    # Charlie-style global normalization.
    spectrum_norm = spectrum - np.nanmin(spectrum)
    denom = np.nanmax(spectrum_norm)
    if np.isfinite(denom) and denom > 0:
        spectrum_norm = spectrum_norm / denom

    if x_axis == "offset":
        if source_x_m is None:
            raise ValueError("source_x_m required for x_axis='offset'.")
        x = receiver_x_m - float(source_x_m)
        xlabel = "Source-receiver offset (m)"
        marker_positions = tuple(float(m) - float(source_x_m) for m in cave_markers_m)
        source_marker_x = 0.0

    elif x_axis == "absolute_offset":
        if source_x_m is None:
            raise ValueError("source_x_m required for x_axis='absolute_offset'.")
        x = np.abs(receiver_x_m - float(source_x_m))
        order = np.argsort(x)
        x = x[order]
        receiver_x_m = receiver_x_m[order]
        working = working[order, :]
        spec = spec[order, :]
        spectrum_norm = spectrum_norm[:, order]
        xlabel = "Absolute source-receiver offset (m)"
        marker_positions = tuple(abs(float(m) - float(source_x_m)) for m in cave_markers_m)
        source_marker_x = 0.0

    elif x_axis == "receiver_x":
        x = receiver_x_m
        xlabel = "Receiver x (m)"
        marker_positions = cave_markers_m
        source_marker_x = source_x_m

    elif x_axis == "trace":
        x = np.arange(receiver_x_m.size, dtype=float)
        xlabel = "Trace number"
        marker_positions = ()
        source_marker_x = None

    else:
        raise ValueError("Invalid x_axis.")

    fig, ax = plt.subplots(figsize=(8.8, 6.5))

    vmin = float(np.nanmin(spectrum_norm))
    vmax = float(np.nanmax(spectrum_norm))
    if vmax <= vmin:
        vmax = vmin + 1.0

    contour_levels = (
        np.linspace(vmin, vmax, levels)
        if isinstance(levels, int)
        else np.asarray(levels, dtype=float)
    )
    norm = Normalize(vmin=vmin, vmax=vmax)

    if plot_type == "contour":
        cs = ax.contour(
            x,
            freqs,
            spectrum_norm,
            levels=contour_levels,
            cmap=cmap,
            norm=norm,
        )
        # A ScalarMappable gives a stable continuous colorbar for line contours.
        mappable = ScalarMappable(norm=norm, cmap=cmap)
        mappable.set_array([])

    elif plot_type == "contourf":
        cs = ax.contourf(
            x,
            freqs,
            spectrum_norm,
            levels=contour_levels,
            cmap=cmap,
            norm=norm,
        )
        mappable = cs

    elif plot_type == "pcolormesh":
        cs = ax.pcolormesh(
            x,
            freqs,
            spectrum_norm,
            shading="auto",
            cmap=cmap,
            norm=norm,
        )
        mappable = cs

    else:
        raise ValueError("plot_type must be 'contour', 'contourf', or 'pcolormesh'.")

    ax.invert_yaxis()
    ax.set_ylim(max_freq_hz, 0)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Frequency (Hz)")
    ax.set_title(title or "Frequency vs offset")
    ax.grid(True, alpha=0.2)

    if line_xlim_m is not None:
        ax.set_xlim(*line_xlim_m)
    else:
        xmin, xmax = float(np.nanmin(x)), float(np.nanmax(x))
        pad = 0.03 * max(xmax - xmin, 1.0)
        ax.set_xlim(xmin - pad, xmax + pad)

    cbar = fig.colorbar(mappable, ax=ax)
    cbar.set_label("Scaled amplitude")

    xmin, xmax = ax.get_xlim()

    if source_marker_x is not None and xmin <= float(source_marker_x) <= xmax:
        ax.axvline(float(source_marker_x), linestyle=":", linewidth=1.2, label="source")

    for xm in marker_positions:
        if xmin <= float(xm) <= xmax:
            ax.axvline(float(xm), linestyle="--", linewidth=1.2, label=f"marker {xm:g} m")

    handles, labels = ax.get_legend_handles_labels()
    if labels:
        unique = dict(zip(labels, handles))
        ax.legend(unique.values(), unique.keys(), loc="best", fontsize=8)

    fig.tight_layout()

    if outfile is not None:
        outfile = Path(outfile)
        outfile.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(outfile, dpi=180, bbox_inches="tight")
        if close_after_save:
            plt.close(fig)

    return {
        "time": time,
        "data": data,
        "processed_data": working,
        "receiver_x_m": receiver_x_m,
        "x": x,
        "x_axis": x_axis,
        "source_x_m": source_x_m,
        "frequencies": freqs,
        "raw_spectrum": spec,
        "spectrum": spectrum_norm,
        "geometry": geom,
        "figure": fig,
        "axis": ax,
        "colorbar": cbar,
    }


## Batch processing

This cell reads each SEG-2 `*.dat` file, assigns provisional geometry, writes SEG-Y, and generates quick-look products.

The outputs are written inside each Geode download folder:

```text
<folder>/wiggle_plots/
<folder>/frequency_contours/charlie_offset/
<folder>/frequency_contours/receiver_x_map/
<folder>/segy/
```


In [5]:
def _frequency_output_paths(
    freq_contour_dir: Path,
    datfile: Path,
    max_freq_hz: float,
) -> tuple[Path, Path]:
    """Return output paths for Charlie-style and receiver-coordinate frequency plots."""
    freq_charlie_file = (
        freq_contour_dir
        / "charlie_offset"
        / f"{datfile.stem}_charlie_frequency_vs_offset_{max_freq_hz:g}Hz.png"
    )
    freq_map_file = (
        freq_contour_dir
        / "receiver_x_map"
        / f"{datfile.stem}_frequency_vs_receiver_x_{max_freq_hz:g}Hz.png"
    )
    return freq_charlie_file, freq_map_file


def make_frequency_contour_products(
    st: Stream,
    geom: dict,
    *,
    title: str,
    freq_contour_dir: Path,
    datfile: Path,
) -> tuple[Path, Path]:
    """Create Charlie-style and receiver-coordinate frequency contour plots."""
    freq_charlie_file, freq_map_file = _frequency_output_paths(
        freq_contour_dir,
        datfile,
        FREQ_CONTOUR_MAX_HZ,
    )

    for parent in [freq_charlie_file.parent, freq_map_file.parent]:
        parent.mkdir(parents=True, exist_ok=True)

    if OVERWRITE_PLOTS or not freq_charlie_file.exists():
        plot_frequency_contour_from_stream_charlie_fft(
            st,
            fallback_receiver_spacing_m=geom["receiver_spacing_m"],
            fallback_first_receiver_x_m=geom["first_receiver_x_m"],
            fallback_source_x_m=geom["source_x_m"],
            source_x_m_override=geom["source_x_m"],
            max_freq_hz=FREQ_CONTOUR_MAX_HZ,
            x_axis="offset",
            plot_type=FREQ_CONTOUR_PLOT_TYPE_CHARLIE,
            levels=FREQ_CONTOUR_LEVELS,
            title=title,# + " — Charlie-style signed-offset plot",
            outfile=freq_charlie_file,
            close_after_save=True,
        )
    else:
        print(f"  Charlie-style frequency contour exists, skipping: {freq_charlie_file.name}")

    if OVERWRITE_PLOTS or not freq_map_file.exists():
        plot_frequency_contour_from_stream_charlie_fft(
            st,
            fallback_receiver_spacing_m=geom["receiver_spacing_m"],
            fallback_first_receiver_x_m=geom["first_receiver_x_m"],
            fallback_source_x_m=geom["source_x_m"],
            source_x_m_override=geom["source_x_m"],
            max_freq_hz=FREQ_CONTOUR_MAX_HZ,
            x_axis="receiver_x",
            line_xlim_m=geom.get("line_xlim_m", None),
            plot_type=FREQ_CONTOUR_PLOT_TYPE_MAP,
            levels=FREQ_CONTOUR_LEVELS,
            title=title + " — receiver-x cave-map plot",
            outfile=freq_map_file,
            cave_markers_m=CAVE_MARKERS_M,
            close_after_save=True,
        )
    else:
        print(f"  receiver-x frequency contour exists, skipping: {freq_map_file.name}")

    return freq_charlie_file, freq_map_file


def process_geode_seg2_file(
    datfile: Path,
    *,
    data_dir: Path,
    metadata_lookup: pd.DataFrame,
    outdir: Path,
    segydir: Path,
    freq_contour_dir: Path,
) -> dict | None:
    """Process one Geode SEG-2 file into plots and optional SEG-Y output.

    Geometry is looked up from the Excel metadata workbook using ``file_no``
    derived from ``datfile.stem``.
    """
    print(f"Processing file: {datfile}")
    st_raw = obspy.read(str(datfile), format="SEG2")

    try:
        file_no = int(datfile.stem)
    except ValueError as exc:
        raise ValueError(f"SEG-2 file stem is not an integer file_no: {datfile.name}") from exc

    meta = lookup_geode_metadata(file_no, metadata_lookup)
    geom = metadata_to_geometry(meta, st_raw)
    st = annotate_geode_stream(st_raw, geom)

    print(
        f"  traces={geom['n_traces']}, duration={geom['duration_s']:.3f} s, "
        f"geom={geom['geometry_name']}, source_x={geom['source_x_m']:.2f} m"
    )

    st_for_wiggle = st
    wiggle_suffix = "shot_gather"
    if APPLY_BANDPASS_FOR_WIGGLES:
        st_for_wiggle = maybe_bandpass(st, *BANDPASS_FREQ_HZ)
        wiggle_suffix += f"_bp_{BANDPASS_FREQ_HZ[0]:g}_{BANDPASS_FREQ_HZ[1]:g}Hz"

    wiggle_file = outdir / f"{datfile.stem}_{wiggle_suffix}.png"
    segy_file = segydir / f"{datfile.stem}.segy"
    freq_charlie_file, freq_map_file = _frequency_output_paths(
        freq_contour_dir,
        datfile,
        FREQ_CONTOUR_MAX_HZ,
    )

    '''
    title = (
        f"{data_dir.name}/{datfile.stem}: source x={geom['source_x_m']:.1f} m, "
        f"{geom['geometry_name']}"
    )
    '''
    
    extras = []

    if geom.get("operator"):
        extras.append(f"operator={geom['operator']}")

    if geom.get("n_blows"):
        extras.append(f"{geom['n_blows']:.0f} blows")

    if geom.get("plate_type"):
        if str(geom["plate_type"]) not in {None, "metal"}:
            extras.append(str(geom["plate_type"]))

    extra_text = ""
    if extras:
        extra_text = " (" + ", ".join(extras) + ")"

    title = (
        f"{data_dir.name}/{datfile.stem}: "
        f"source x={geom['source_x_m']:.1f} m, "
        f"{geom['geometry_name']}"
        f"{extra_text}"
    )

    if OVERWRITE_PLOTS or not wiggle_file.exists():
        plot_wiggle_gather_from_stream(
            st_for_wiggle,
            fallback_receiver_spacing_m=geom["receiver_spacing_m"],
            fallback_first_receiver_x_m=geom["first_receiver_x_m"],
            fallback_source_x_m=geom["source_x_m"],
            tmin=0.0,
            tmax=geom["duration_s"],
            scale=wiggle_scale,
            normalize=normalize_wiggles,
            title=title,
            outfile=wiggle_file,
        )
    else:
        print(f"  wiggle exists, skipping: {wiggle_file.name}")

    if MAKE_FREQUENCY_CONTOURS:
        freq_charlie_file, freq_map_file = make_frequency_contour_products(
            st,
            geom,
            title=title,
            freq_contour_dir=freq_contour_dir,
            datfile=datfile,
        )

    if WRITE_SEGY:
        if OVERWRITE_SEGY or not segy_file.exists():
            # ObsPy's direct SEG-Y writer is retained here because these are raw field SEG-2
            # files without finalized geometry headers. Later, replace this with a metadata-aware
            # segy_tools export that writes authoritative source/receiver coordinates.
            st.write(str(segy_file), format="SEGY", byteorder=">")
        else:
            print(f"  SEG-Y exists, skipping: {segy_file.name}")

    return {
        "folder": data_dir.name,
        "datfile": datfile.name,
        "n_traces": geom["n_traces"],
        "duration_s": geom["duration_s"],
        "file_no": geom["file_no"],
        "sheet_name": geom.get("sheet_name"),
        "transect": geom.get("transect"),
        "survey": geom.get("survey"),
        "geometry_name": geom["geometry_name"],
        "receiver_spacing_m": geom["receiver_spacing_m"],
        "first_receiver_x_m": geom["first_receiver_x_m"],
        "last_receiver_x_m": geom.get("last_receiver_x_m"),
        "source_x_m": geom["source_x_m"],
        "n_blows": geom.get("n_blows"),
        "operator": geom.get("operator"),
        "plate_type": geom.get("plate_type"),
        "confidence": geom.get("confidence"),
        "review_status": geom.get("review_status"),
        "comment": geom.get("comment"),
        "wiggle_file": str(wiggle_file),
        "freq_charlie_file": str(freq_charlie_file) if MAKE_FREQUENCY_CONTOURS else None,
        "freq_map_file": str(freq_map_file) if MAKE_FREQUENCY_CONTOURS else None,
        "segy_file": str(segy_file) if WRITE_SEGY else None,
    }


def process_geode_folders(
    download_dir: Path,
    prefixes=("LBSSP_",),
    *,
    metadata_xlsx: Path = METADATA_XLSX,
    metadata_sheets: list[str] | tuple[str, ...] = METADATA_SHEETS,
) -> pd.DataFrame:
    """Batch-process all matching Geode folders and return a summary table."""
    geode_folders = find_geode_folders(download_dir, prefixes=prefixes)
    print(f"Found {len(geode_folders)} matching Geode folders")

    metadata_lookup = load_geode_metadata_lookup(metadata_xlsx, metadata_sheets)

    records = []
    for day_index, data_dir in enumerate(geode_folders):
        field_date = parse_date_from_geode_folder(data_dir)
        print("\n" + "=" * 80)
        print(f"Folder: {data_dir}")
        print(f"Parsed date: {field_date}")

        alldatfiles = sorted(data_dir.glob("*.dat"))
        print(f"Found {len(alldatfiles)} SEG-2 .dat files")

        outdir = data_dir / "wiggle_plots"
        segydir = data_dir / "segy"
        freq_contour_dir = data_dir / "frequency_contours"
        for d in [outdir, segydir, freq_contour_dir]:
            d.mkdir(exist_ok=True)

        for file_index, datfile in enumerate(alldatfiles):
            try:
                rec = process_geode_seg2_file(
                    datfile,
                    data_dir=data_dir,
                    metadata_lookup=metadata_lookup,
                    outdir=outdir,
                    segydir=segydir,
                    freq_contour_dir=freq_contour_dir,
                )
                if rec is None:
                    print(f"  skipped file {datfile.name} due to geometry assumptions")
                    continue
                rec["field_date"] = field_date.isoformat() if field_date else None
                records.append(rec)
            except Exception as exc:
                print(f"ERROR processing {datfile}: {exc}")
                records.append({
                    "folder": data_dir.name,
                    "datfile": datfile.name,
                    "error": repr(exc),
                })

    summary = pd.DataFrame(records)
    if len(summary):
        summary_file = download_dir / "geode_shot_gather_processing_summary.csv"
        summary.to_csv(summary_file, index=False)
        print(f"\nWrote summary: {summary_file}")
    return summary


## Run batch processing

Uncomment and run the cell below when ready. The summary table records the assumed geometry and all output products.


In [6]:
summary = process_geode_folders(DOWNLOAD_DIR, prefixes=FOLDER_PREFIXES)
summary.head()


Found 2 matching Geode folders
Loaded 327 metadata rows from /Users/glennthompson/Library/CloudStorage/Box-Box/thompsong/2026KarstGeophysicsDEP/04_FinalReport/seismic_analysis/jochen_field_notes_metadata_tables.xlsx

Folder: /Users/glennthompson/Downloads/LBSSP_051826
Parsed date: 2026-05-18
Found 84 SEG-2 .dat files
Processing file: /Users/glennthompson/Downloads/LBSSP_051826/3005.dat
  traces=72, duration=0.400 s, geom=T1_1m_refraction, source_x=82.50 m


/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/io/seg2/seg2.py:365: UserWarning: Many companies use custom defined SEG2 header variables. This might cause basic header information reflected in the single traces' stats to be wrong (e.g. recording delays, first sample number, station code names, ..). Please check the complete list of additional unmapped header fields that gets stored in Trace.stats.seg2 and/or the manual of the source of the SEG2 files for fields that might influence e.g. trace start times.
  warnings.warn(WARNING_HEADER)


  SEG-Y exists, skipping: 3005.segy
Processing file: /Users/glennthompson/Downloads/LBSSP_051826/3006.dat
  traces=72, duration=0.400 s, geom=T1_1m_refraction, source_x=84.50 m
  SEG-Y exists, skipping: 3006.segy
Processing file: /Users/glennthompson/Downloads/LBSSP_051826/3007.dat
  traces=72, duration=0.400 s, geom=T1_1m_refraction, source_x=86.50 m
  SEG-Y exists, skipping: 3007.segy
Processing file: /Users/glennthompson/Downloads/LBSSP_051826/3008.dat
  traces=72, duration=0.400 s, geom=T1_1m_refraction, source_x=88.50 m
  SEG-Y exists, skipping: 3008.segy
Processing file: /Users/glennthompson/Downloads/LBSSP_051826/3009.dat
  traces=72, duration=0.400 s, geom=T1_1m_refraction, source_x=90.50 m
  SEG-Y exists, skipping: 3009.segy
Processing file: /Users/glennthompson/Downloads/LBSSP_051826/3010.dat
  traces=72, duration=0.400 s, geom=T1_1m_refraction, source_x=92.50 m
  SEG-Y exists, skipping: 3010.segy
Processing file: /Users/glennthompson/Downloads/LBSSP_051826/3011.dat
  traces=

/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/io/segy/core.py:362: UserWarning: CREATING TRACE HEADER
  warnings.warn("CREATING TRACE HEADER")


Processing file: /Users/glennthompson/Downloads/LBSSP_051826/3048.dat
  traces=72, duration=0.600 s, geom=T1_2m_refraction, source_x=47.00 m
Processing file: /Users/glennthompson/Downloads/LBSSP_051826/3049.dat
  traces=72, duration=0.600 s, geom=T1_2m_refraction, source_x=51.00 m
Processing file: /Users/glennthompson/Downloads/LBSSP_051826/3050.dat
  traces=72, duration=0.600 s, geom=T1_2m_refraction, source_x=55.00 m
Processing file: /Users/glennthompson/Downloads/LBSSP_051826/3051.dat
  traces=72, duration=0.600 s, geom=T1_2m_refraction, source_x=59.00 m
Processing file: /Users/glennthompson/Downloads/LBSSP_051826/3052.dat
  traces=72, duration=0.600 s, geom=T1_2m_refraction, source_x=63.00 m
Processing file: /Users/glennthompson/Downloads/LBSSP_051826/3053.dat
  traces=72, duration=0.600 s, geom=T1_2m_refraction, source_x=67.00 m
Processing file: /Users/glennthompson/Downloads/LBSSP_051826/3054.dat
  traces=72, duration=0.600 s, geom=T1_2m_refraction, source_x=71.00 m
Processing fi

,folder,datfile,n_traces,duration_s,file_no,sheet_name,transect,survey,geometry_name,receiver_spacing_m,...,plate_type,confidence,review_status,comment,wiggle_file,freq_charlie_file,freq_map_file,segy_file,field_date,error
0,LBSSP_051826,3005.dat,72.0,0.4,3005.0,T1_1m_Refraction,T1,T1_1m_refraction,T1_1m_refraction,1.0,...,metal,Medium,CHECK,delayed; first standard-looking shot,/Users/glennthompson/Downloads/LBSSP_051826/wi...,/Users/glennthompson/Downloads/LBSSP_051826/fr...,/Users/glennthompson/Downloads/LBSSP_051826/fr...,/Users/glennthompson/Downloads/LBSSP_051826/se...,2026-05-18,NaN
1,LBSSP_051826,3006.dat,72.0,0.4,3006.0,T1_1m_Refraction,T1,T1_1m_refraction,T1_1m_refraction,1.0,...,metal,High,OK,None,/Users/glennthompson/Downloads/LBSSP_051826/wi...,/Users/glennthompson/Downloads/LBSSP_051826/fr...,/Users/glennthompson/Downloads/LBSSP_051826/fr...,/Users/glennthompson/Downloads/LBSSP_051826/se...,2026-05-18,NaN
2,LBSSP_051826,3007.dat,72.0,0.4,3007.0,T1_1m_Refraction,T1,T1_1m_refraction,T1_1m_refraction,1.0,...,metal,Low,CHECK,None,/Users/glennthompson/Downloads/LBSSP_051826/wi...,/Users/glennthompson/Downloads/LBSSP_051826/fr...,/Users/glennthompson/Downloads/LBSSP_051826/fr...,/Users/glennthompson/Downloads/LBSSP_051826/se...,2026-05-18,NaN
3,LBSSP_051826,3008.dat,72.0,0.4,3008.0,T1_1m_Refraction,T1,T1_1m_refraction,T1_1m_refraction,1.0,...,metal,High,OK,None,/Users/glennthompson/Downloads/LBSSP_051826/wi...,/Users/glennthompson/Downloads/LBSSP_051826/fr...,/Users/glennthompson/Downloads/LBSSP_051826/fr...,/Users/glennthompson/Downloads/LBSSP_051826/se...,2026-05-18,NaN
4,LBSSP_051826,3009.dat,72.0,0.4,3009.0,T1_1m_Refraction,T1,T1_1m_refraction,T1_1m_refraction,1.0,...,metal,High,OK,None,/Users/glennthompson/Downloads/LBSSP_051826/wi...,/Users/glennthompson/Downloads/LBSSP_051826/fr...,/Users/glennthompson/Downloads/LBSSP_051826/fr...,/Users/glennthompson/Downloads/LBSSP_051826/se...,2026-05-18,NaN


## Inspect processing summary

Use this cell to identify failed files, check geometry assumptions, and quickly locate generated frequency contour plots.


In [7]:
if "summary" in globals() and len(summary):
    display(summary)
    if "error" in summary.columns:
        display(summary[summary["error"].notna()])


,folder,datfile,n_traces,duration_s,file_no,sheet_name,transect,survey,geometry_name,receiver_spacing_m,...,plate_type,confidence,review_status,comment,wiggle_file,freq_charlie_file,freq_map_file,segy_file,field_date,error
0,LBSSP_051826,3005.dat,72.0,0.4,3005.0,T1_1m_Refraction,T1,T1_1m_refraction,T1_1m_refraction,1.0,...,metal,Medium,CHECK,delayed; first standard-looking shot,/Users/glennthompson/Downloads/LBSSP_051826/wi...,/Users/glennthompson/Downloads/LBSSP_051826/fr...,/Users/glennthompson/Downloads/LBSSP_051826/fr...,/Users/glennthompson/Downloads/LBSSP_051826/se...,2026-05-18,NaN
1,LBSSP_051826,3006.dat,72.0,0.4,3006.0,T1_1m_Refraction,T1,T1_1m_refraction,T1_1m_refraction,1.0,...,metal,High,OK,None,/Users/glennthompson/Downloads/LBSSP_051826/wi...,/Users/glennthompson/Downloads/LBSSP_051826/fr...,/Users/glennthompson/Downloads/LBSSP_051826/fr...,/Users/glennthompson/Downloads/LBSSP_051826/se...,2026-05-18,NaN
2,LBSSP_051826,3007.dat,72.0,0.4,3007.0,T1_1m_Refraction,T1,T1_1m_refraction,T1_1m_refraction,1.0,...,metal,Low,CHECK,None,/Users/glennthompson/Downloads/LBSSP_051826/wi...,/Users/glennthompson/Downloads/LBSSP_051826/fr...,/Users/glennthompson/Downloads/LBSSP_051826/fr...,/Users/glennthompson/Downloads/LBSSP_051826/se...,2026-05-18,NaN
3,LBSSP_051826,3008.dat,72.0,0.4,3008.0,T1_1m_Refraction,T1,T1_1m_refraction,T1_1m_refraction,1.0,...,metal,High,OK,None,/Users/glennthompson/Downloads/LBSSP_051826/wi...,/Users/glennthompson/Downloads/LBSSP_051826/fr...,/Users/glennthompson/Downloads/LBSSP_051826/fr...,/Users/glennthompson/Downloads/LBSSP_051826/se...,2026-05-18,NaN
4,LBSSP_051826,3009.dat,72.0,0.4,3009.0,T1_1m_Refraction,T1,T1_1m_refraction,T1_1m_refraction,1.0,...,metal,High,OK,None,/Users/glennthompson/Downloads/LBSSP_051826/wi...,/Users/glennthompson/Downloads/LBSSP_051826/fr...,/Users/glennthompson/Downloads/LBSSP_051826/fr...,/Users/glennthompson/Downloads/LBSSP_051826/se...,2026-05-18,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149,LBSSP_051926,4066.dat,72.0,0.4,4066.0,T4_1m_Refraction,T4,T4_1m_refraction,T4_1m_refraction,1.0,...,metal,High,OK,None,/Users/glennthompson/Downloads/LBSSP_051926/wi...,/Users/glennthompson/Downloads/LBSSP_051926/fr...,/Users/glennthompson/Downloads/LBSSP_051926/fr...,/Users/glennthompson/Downloads/LBSSP_051926/se...,2026-05-19,NaN
150,LBSSP_051926,4067.dat,72.0,0.4,4067.0,T4_1m_Refraction,T4,T4_1m_refraction,T4_1m_refraction,1.0,...,metal,High,OK,None,/Users/glennthompson/Downloads/LBSSP_051926/wi...,/Users/glennthompson/Downloads/LBSSP_051926/fr...,/Users/glennthompson/Downloads/LBSSP_051926/fr...,/Users/glennthompson/Downloads/LBSSP_051926/se...,2026-05-19,NaN
151,LBSSP_051926,4068.dat,72.0,0.4,4068.0,T4_1m_Refraction,T4,T4_1m_refraction,T4_1m_refraction,1.0,...,metal,High,OK,None,/Users/glennthompson/Downloads/LBSSP_051926/wi...,/Users/glennthompson/Downloads/LBSSP_051926/fr...,/Users/glennthompson/Downloads/LBSSP_051926/fr...,/Users/glennthompson/Downloads/LBSSP_051926/se...,2026-05-19,NaN
152,LBSSP_051926,4069.dat,72.0,0.4,4069.0,T4_1m_Refraction,T4,T4_1m_refraction,T4_1m_refraction,1.0,...,metal,High,OK,None,/Users/glennthompson/Downloads/LBSSP_051926/wi...,/Users/glennthompson/Downloads/LBSSP_051926/fr...,/Users/glennthompson/Downloads/LBSSP_051926/fr...,/Users/glennthompson/Downloads/LBSSP_051926/se...,2026-05-19,NaN


,folder,datfile,n_traces,duration_s,file_no,sheet_name,transect,survey,geometry_name,receiver_spacing_m,...,plate_type,confidence,review_status,comment,wiggle_file,freq_charlie_file,freq_map_file,segy_file,field_date,error
139,LBSSP_051926,4056.dat,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,KeyError('No metadata found for file_no=4056')
